In [2]:
import os
from pathlib import Path

HOME_DIR = os.path.dirname(os.getcwd())
DATA_DIR = Path("/data/dacon-236105")

# EDA 1. Audio Data Basic Statistics

In [3]:
from glob import glob
from IPython.display import display
import natsort
import numpy as np
import os
import pandas as pd
from scipy.io import wavfile

## Audio Files

### File Names

In [6]:
train_wav_fpaths = natsort.natsorted(glob(os.path.join(DATA_DIR, "train", "*.wav")))
test_wav_fpaths = natsort.natsorted(glob(os.path.join(DATA_DIR, "test", "*.wav")))

len(train_wav_fpaths), len(test_wav_fpaths)

(5001, 1881)

### Basic Auditory Information

In [11]:
def get_audio_scale(y):
    dtype = y.dtype

    if np.issubdtype(dtype, np.floating): return 1.0

    elif np.issubdtype(dtype, np.integer):
        if dtype == np.int8: raise TypeError(f"Unsupported dtype: {dtype}")

        info = np.iinfo(dtype)
        limit = np.abs(info.min)

        if dtype == np.int32:
            limit_24bit = 2 ** 23
            if np.max(np.abs(y)) <= limit_24bit: return limit_24bit

        return limit

    raise TypeError(f"Unsupported dtype: {dtype}")

def rms(y): return np.sqrt(np.mean(y.astype(float)**2))

def dbfs(y): return 20 * np.log10(rms(y) / get_audio_scale(y) + 1e-9)

In [ ]:
.id

0       TRAIN_0000
1       TRAIN_0001
2       TRAIN_0002
3       TRAIN_0003
4       TRAIN_0004
           ...    
4996    TRAIN_4996
4997    TRAIN_4997
4998    TRAIN_4998
4999    TRAIN_4999
5000    TRAIN_5000
Name: id, Length: 5001, dtype: object

In [ ]:
data = []

for wav_fpath in train_wav_fpaths:
    sr, y = wavfile.read(wav_fpath)
    data.append({
        "id": os.path.basename(wav_fpath).replace(".wav", ''),

        # Basic Info
        "sample_rate": sr,                              # sampling rate
        "bit_depth": y.dtype,                           # int16, int32, float32, etc.
        "channel": 1 if y.ndim == 1 else y.shape[1],    # 1 for mono, 2 for stereo, etc.
        
        # Duration
        "len": len(y),                                  # raw sample length
        "dur_sec": y.shape[0] / sr,                     # duration in seconds

        # Amplitude
        "amp_min": np.min(y),                           # if -32768, some clipping may have occurred
        "amp_max": np.max(y),                           # if 32767, some clipping may have occurred

        # Signal Level
        "rms": rms(y),                                  # root mean square
        "dbfs": dbfs(y),                                # deciBels relative to Full Scale
    })

train_df = pd.DataFrame(data).merge(
    pd.read_csv(os.path.join(DATA_DIR, "train.csv")),
    on="id", how="inner"
)
del wav_fpath, sr, y, data
display(train_df.head(10))

,id,sample_rate,bit_depth,channel,len,dur_sec,amp_min,amp_max,rms,dbfs,path,label
0,TRAIN_0000,16000,int16,1,37904,2.369000,-1364,1370,223.039790,-43.341350,./train/TRAIN_0000.wav,1
1,TRAIN_0001,16000,int16,1,41642,2.602625,-1517,1149,259.496729,-42.026360,./train/TRAIN_0001.wav,2
2,TRAIN_0002,16000,int16,1,46446,2.902875,-6442,7420,910.593334,-31.122509,./train/TRAIN_0002.wav,4
3,TRAIN_0003,16000,int16,1,36837,2.302313,-7604,7286,959.786399,-30.665507,./train/TRAIN_0003.wav,5
4,TRAIN_0004,16000,int16,1,29897,1.868562,-14531,17987,1326.059141,-27.857741,./train/TRAIN_0004.wav,4
5,TRAIN_0005,16000,int16,1,40040,2.502500,-2476,2813,408.389290,-38.087511,./train/TRAIN_0005.wav,3
6,TRAIN_0006,16000,int16,1,33100,2.068750,-32768,32767,6683.834210,-13.808485,./train/TRAIN_0006.wav,0
7,TRAIN_0007,16000,int16,1,36837,2.302313,-7364,8369,849.018204,-31.730658,./train/TRAIN_0007.wav,0
8,TRAIN_0008,16000,int16,1,41642,2.602625,-1187,1286,265.569793,-41.825424,./train/TRAIN_0008.wav,2
9,TRAIN_0009,16000,int16,1,35235,2.202187,-909,970,167.222482,-45.843104,./train/TRAIN_0009.wav,1


In [32]:
data = []

for wav_fpath in test_wav_fpaths:
    sr, y = wavfile.read(wav_fpath)
    data.append({
        "id": os.path.basename(wav_fpath).replace(".wav", ''),

        # Basic Info
        "sample_rate": sr,                              # sampling rate
        "bit_depth": y.dtype,                           # int16, int32, float32, etc.
        "channel": 1 if y.ndim == 1 else y.shape[1],    # 1 for mono, 2 for stereo, etc.
        
        # Duration
        "len": len(y),                                  # raw sample length
        "dur_sec": y.shape[0] / sr,                     # duration in seconds

        # Amplitude
        "amp_min": np.min(y),                           # if -32768, some clipping may have occurred
        "amp_max": np.max(y),                           # if 32767, some clipping may have occurred

        # Signal Level
        "rms": rms(y),                                  # root mean square
        "dbfs": dbfs(y),                                # deciBels relative to Full Scale
    })

test_df = pd.DataFrame(data).merge(
    pd.read_csv(os.path.join(DATA_DIR, "test.csv")),
    on="id", how="inner"
)
del wav_fpath, sr, y, data
display(test_df.head())

,id,sample_rate,bit_depth,channel,len,dur_sec,amp_min,amp_max,rms,dbfs,path
0,TEST_0000,16000,int16,1,38438,2.402375,-13234,12183,1677.006378,-25.818304,./test/TEST_0000.wav
1,TEST_0001,16000,int16,1,41642,2.602625,-21130,25898,3442.870271,-19.570585,./test/TEST_0001.wav
2,TEST_0002,16000,int16,1,43777,2.736063,-1283,1901,290.840999,-41.035885,./test/TEST_0002.wav
3,TEST_0003,16000,int16,1,43243,2.702688,-4470,5449,557.039883,-35.391272,./test/TEST_0003.wav
4,TEST_0004,16000,int16,1,29896,1.868500,-15188,16689,2202.454288,-23.450861,./test/TEST_0004.wav


In [14]:
train_df.sample_rate.unique(), test_df.sample_rate.unique()

(array([16000]), array([16000]))

In [15]:
train_df.bit_depth.unique(), test_df.bit_depth.unique()

(array([dtype('int16')], dtype=object), array([dtype('int16')], dtype=object))

In [16]:
train_df.channel.unique(), test_df.channel.unique()

(array([1]), array([1]))

In [17]:
display(train_df.dur_sec.describe())
display(test_df.dur_sec.describe())

count    5001.000000
mean        2.542619
std         0.496056
min         1.334688
25%         2.202187
50%         2.502500
75%         2.836187
max         4.871562
Name: dur_sec, dtype: float64

count    1881.000000
mean        2.541450
std         0.530869
min         1.334688
25%         2.168813
50%         2.469125
75%         2.836187
max         5.005000
Name: dur_sec, dtype: float64

In [18]:
display(train_df.dbfs.describe())
display(test_df.dbfs.describe())

count    5001.000000
mean      -31.651887
std         7.690457
min      -180.000000
25%       -37.326280
50%       -32.910147
75%       -26.639937
max       -10.619520
Name: dbfs, dtype: float64

count    1881.000000
mean      -31.590734
std         7.267106
min       -47.579837
25%       -37.116283
50%       -32.690854
75%       -26.920915
max       -10.356399
Name: dbfs, dtype: float64

In [33]:
train_df["label"].value_counts()

label
0    867
2    859
5    853
3    852
1    848
4    722
Name: count, dtype: int64